This Notebook combines all raw expression and signaling data into datasets that can be used by the IL-10 models.

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from scipy.optimize import curve_fit
from matplotlib import use
use('Agg')
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)
from sklearn.linear_model import TheilSenRegressor
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
import scipy.stats as stats

## Signaling data

#### Data from Groby et al., 2020 (Monocytes)

In [12]:
df = pd.DataFrame(columns=["Plot","Cell_type","Variant","STAT_type","IL","pSTAT"])
df_WT_IL10 = pd.read_csv("data/signaling/raw/IL10_WT.csv", index_col=0)
df_mut_IL10 = pd.read_csv("data/signaling/raw/IL10_mut.csv", index_col=0)
df_WT_STAT = pd.read_csv("data/signaling/raw/STAT_WT.csv", index_col=0)
df_mut_STAT = pd.read_csv("data/signaling/raw/STAT_mut.csv", index_col=0)
df["IL"] = df_WT_IL10.melt()["value"].to_list()+df_mut_IL10.melt()["value"].to_list()
df["Cell_type"] = df_WT_IL10.melt()["variable"].to_list()+df_mut_IL10.melt()["variable"].to_list()
df["pSTAT"] = df_WT_STAT.melt()["value"].to_list()+df_mut_STAT.melt()["value"].to_list()
mapping = {value: idx for idx, value in enumerate(set(df["Cell_type"]))}
df["Plot"] = np.array([mapping[value] for value in df["Cell_type"]]) + 1
df["Variant"] = ["WT"] * len(df_WT_IL10.melt()["variable"].to_list()) + ["R5A11D"] * len(df_WT_IL10.melt()["variable"].to_list())
df = df.loc[df["Cell_type"].str.contains("Gorby")]
df["STAT_type"] = ["p"+elem.split("_")[1] for elem in df["Cell_type"]]
df["Cell_type"] = [elem.split("_")[0] for elem in df["Cell_type"]]
df = df.dropna()
for i in range (0,len(df['Plot'].drop_duplicates().values)):
    df['Plot'] = df['Plot'].replace(df['Plot'].drop_duplicates().values[i], i+1)
df = df.reset_index(drop=True)
df["Cell_type"] = "MO.Mean"

#### Data from Saxton et al., 2021

In [3]:
cells = ["B","Daudi","Jurkat","Monocytes","NK","T4","T8","THP1","YT1"]
cells_new = ["B.Mean","ACH-000786","ACH-000995","MO.Mean","NK","T4.Mean","T8.Mean","ACH-000146","ACH-002317"]
plot_num = 3
cell_i = 0
for cell in cells:
    df_cell = pd.DataFrame(columns=["Plot","Cell_type","Variant","STAT_type","IL","pSTAT"])
    csv_files = [f for f in os.listdir("data/signaling/raw/"+cell) if f.endswith(".csv")]
    for csv_file in csv_files:
        df_mut = pd.read_csv("data/signaling/raw/"+cell+"/"+csv_file, header=None)
        df_mut.columns = ["IL","pSTAT"]
        df_mut["Plot"] = plot_num
        df_mut["Cell_type"] = cells_new[cell_i]
        df_mut["Variant"] = csv_file.split(".csv")[0]
        df_mut["STAT_type"] = "pSTAT3"
        df = pd.concat([df, df_mut], ignore_index=True)
    plot_num += 1
    cell_i += 1

In [4]:
for plot_num in df["Plot"].drop_duplicates():
    df.loc[df["Plot"]==plot_num,"pSTAT"] = df.loc[df["Plot"]==plot_num,"pSTAT"]-df.loc[df["Plot"]==plot_num,"pSTAT"].min()

#### Get EC50 and amplitude from dose-response curves

In [3]:
def hill_func(inp,B,K):
    out=B*inp/(K+inp)
    return out

In [6]:
df_EC50 = pd.DataFrame(columns=['Plot','Cell_type','Variant','STAT_type','EC50','Amp'])
for plot_num in df["Plot"].drop_duplicates():
    df_plot = df.loc[df["Plot"]==plot_num]
    mutant_list = df_plot["Variant"].drop_duplicates().to_list()
    mutant_list.remove("WT")
    
    fit, cov = curve_fit(hill_func, df_plot.loc[df_plot["Variant"]=="WT","IL"].values, df_plot.loc[df_plot["Variant"]=="WT","pSTAT"].values, bounds = ([0,df_plot.loc[df_plot["Variant"]=="WT","IL"].min()/10], [df_plot.loc[df_plot["Variant"]=="WT","pSTAT"].max(), df_plot.loc[df_plot["Variant"]=="WT","IL"].max()*10]))
    mut_WT = fit[0]
    df_EC50.loc[len(df_EC50)] = [plot_num,df_plot["Cell_type"].values[0],"WT",df_plot["STAT_type"].values[0],np.log10(fit[1]),fit[0]/mut_WT*100]
    df.loc[df["Plot"]==plot_num,"pSTAT"] = df.loc[df["Plot"]==plot_num,"pSTAT"]/mut_WT*100
    for mutant in mutant_list:
        fit, cov = curve_fit(hill_func, df_plot.loc[df_plot["Variant"]==mutant,"IL"].values, df_plot.loc[df_plot["Variant"]==mutant,"pSTAT"].values, bounds = ([0,df_plot.loc[df_plot["Variant"]==mutant,"IL"].min()/10], [df_plot.loc[df_plot["Variant"]==mutant,"pSTAT"].max(), df_plot.loc[df_plot["Variant"]==mutant,"IL"].max()*10]))
        df_EC50.loc[len(df_EC50)] = [plot_num,df_plot["Cell_type"].values[0],mutant,df_plot["STAT_type"].values[0],np.log10(fit[1]),fit[0]/mut_WT*100]
df.to_csv("data/signaling/IL10_STAT_data.tsv.gz", sep="\t", index=False, compression="gzip") # After normalizing valeus to the amplitude we save the signaling dataset

Add Amplitude data from Saxton et al. (STAT1)

In [8]:
df_STAT1 = pd.read_csv("data/signaling/raw/AMP_STAT1.csv", header=None)[[1]]
df_STAT1.columns = ["Amp"]
df_STAT1["Plot"] = [12,12,12,13,13,13]
df_STAT1["Cell_type"] = ["T8.Mean","T8.Mean","T8.Mean","T4.Mean","T4.Mean","T4.Mean"]
df_STAT1["Variant"] = ["WT","Super-10","10-DE","WT","Super-10","10-DE"]
df_STAT1["STAT_type"] = "pSTAT1"
df_STAT1["EC50"] = np.nan
df_EC50 = pd.concat([df_EC50, df_STAT1], ignore_index=True)
df_EC50["EC50_m"] = np.nan
df_EC50["Amp_m"] = np.nan
df_EC50["EC50_e"] = np.nan
df_EC50["Amp_e"] = np.nan
df_EC50["Error"] = np.nan
df_EC50.to_csv("data/signaling/IL10_EC50_AMP.tsv.gz", sep="\t", index=False, compression="gzip")    

#### Data from Groby et al., 2020 (CD8+ T cells)

In [17]:
df_WT = pd.read_csv("data/signaling/raw/T8_S3_WT.csv", header=None)
df_WT.columns = ["IL","pSTAT"]
df_WT["Plot"] = 14
df_WT["Cell_type"] = "T8.Mean"
df_WT["Variant"] = "WT"
df_WT["STAT_type"] = "pSTAT3"
df_mut = pd.read_csv("data/signaling/raw/T8_S3_R5A11D.csv", header=None)
df_mut.columns = ["IL","pSTAT"]
df_mut["Plot"] = 14
df_mut["Cell_type"] = "T8.Mean"
df_mut["Variant"] = "R5A11D"
df_mut["STAT_type"] = "pSTAT3"
df = pd.concat([df_WT, df_mut], ignore_index=True)
df_WT = pd.read_csv("data/signaling/raw/T8_S1_WT.csv", header=None)
df_WT.columns = ["IL","pSTAT"]
df_WT["Plot"] = 15
df_WT["Cell_type"] = "T8.Mean"
df_WT["Variant"] = "WT"
df_WT["STAT_type"] = "pSTAT1"
df = pd.concat([df, df_WT], ignore_index=True)
df_mut = pd.read_csv("data/signaling/raw/T8_S1_R5A11D.csv", header=None)
df_mut.columns = ["IL","pSTAT"]
df_mut["Plot"] = 15
df_mut["Cell_type"] = "T8.Mean"
df_mut["Variant"] = "R5A11D"
df_mut["STAT_type"] = "pSTAT1"
df = pd.concat([df, df_mut], ignore_index=True)
df = df[["Plot","Cell_type","Variant","STAT_type","IL","pSTAT"]]

In [18]:
df_EC50 = pd.DataFrame(columns=['Plot','Cell_type','Variant','STAT_type','EC50','Amp'])
for plot_num in df["Plot"].drop_duplicates():
    df_plot = df.loc[df["Plot"]==plot_num]
    mutant_list = df_plot["Variant"].drop_duplicates().to_list()
    mutant_list.remove("WT")
    
    fit, cov = curve_fit(hill_func, df_plot.loc[df_plot["Variant"]=="WT","IL"].values, df_plot.loc[df_plot["Variant"]=="WT","pSTAT"].values, bounds = ([0,df_plot.loc[df_plot["Variant"]=="WT","IL"].min()/10], [df_plot.loc[df_plot["Variant"]=="WT","pSTAT"].max(), df_plot.loc[df_plot["Variant"]=="WT","IL"].max()*10]))
    mut_WT = fit[0]
    df_EC50.loc[len(df_EC50)] = [plot_num,df_plot["Cell_type"].values[0],"WT",df_plot["STAT_type"].values[0],np.log10(fit[1]),fit[0]/mut_WT*100]
    df.loc[df["Plot"]==plot_num,"pSTAT"] = df.loc[df["Plot"]==plot_num,"pSTAT"]/mut_WT*100
    for mutant in mutant_list:
        fit, cov = curve_fit(hill_func, df_plot.loc[df_plot["Variant"]==mutant,"IL"].values, df_plot.loc[df_plot["Variant"]==mutant,"pSTAT"].values, bounds = ([0,df_plot.loc[df_plot["Variant"]==mutant,"IL"].min()/10], [df_plot.loc[df_plot["Variant"]==mutant,"pSTAT"].max(), df_plot.loc[df_plot["Variant"]==mutant,"IL"].max()*10]))
        df_EC50.loc[len(df_EC50)] = [plot_num,df_plot["Cell_type"].values[0],mutant,df_plot["STAT_type"].values[0],np.log10(fit[1]),fit[0]/mut_WT*100]
df.to_csv("data/signaling/IL10_STAT_data_CD8_Gorby.tsv.gz", sep="\t", index=False, compression="gzip") # After normalizing valeus to the amplitude we save the signaling dataset
df_EC50.to_csv("data/signaling/IL10_EC50_AMP_CD8_Gorby.tsv.gz", sep="\t", index=False, compression="gzip")

#### Data from Groby et al., 2020 (Monomeric IL-10)

In [5]:
df = pd.DataFrame(columns=["Plot","Cell_type","Variant","STAT_type","IL","pSTAT"])
cells = ["Monocytes","T8"]
cells_new = ["MO.Mean","T8.Mean"]
plot_num = 1
cell_i = 0
for cell in cells:
    df_cell = pd.DataFrame(columns=["Plot","Cell_type","Variant","STAT_type","IL","pSTAT"])
    csv_files = [f for f in os.listdir("data/signaling/raw/Monomeric_IL10/S3/"+cell) if f.endswith(".csv")]
    for csv_file in csv_files:
        df_mut = pd.read_csv("data/signaling/raw/Monomeric_IL10/S3/"+cell+"/"+csv_file, header=None)
        df_mut.columns = ["IL","pSTAT"]
        df_mut["Plot"] = plot_num
        df_mut["Cell_type"] = cells_new[cell_i]
        df_mut["Variant"] = csv_file.split(".csv")[0]
        df_mut["STAT_type"] = "pSTAT3"
        df = pd.concat([df, df_mut], ignore_index=True)
    plot_num += 1
    cell_i += 1
cell_i = 0
for cell in cells:
    df_cell = pd.DataFrame(columns=["Plot","Cell_type","Variant","STAT_type","IL","pSTAT"])
    csv_files = [f for f in os.listdir("data/signaling/raw/Monomeric_IL10/S1/"+cell) if f.endswith(".csv")]
    for csv_file in csv_files:
        df_mut = pd.read_csv("data/signaling/raw/Monomeric_IL10/S1/"+cell+"/"+csv_file, header=None)
        df_mut.columns = ["IL","pSTAT"]
        df_mut["Plot"] = plot_num
        df_mut["Cell_type"] = cells_new[cell_i]
        df_mut["Variant"] = csv_file.split(".csv")[0]
        df_mut["STAT_type"] = "pSTAT1"
        df = pd.concat([df, df_mut], ignore_index=True)
    plot_num += 1
    cell_i += 1

In [7]:
df_EC50 = pd.DataFrame(columns=['Plot','Cell_type','Variant','STAT_type','EC50','Amp'])
for plot_num in df["Plot"].drop_duplicates():
    df_plot = df.loc[df["Plot"]==plot_num]
    mutant_list = df_plot["Variant"].drop_duplicates().to_list()
    mutant_list.remove("WT")
    
    fit, cov = curve_fit(hill_func, df_plot.loc[df_plot["Variant"]=="WT","IL"].values, df_plot.loc[df_plot["Variant"]=="WT","pSTAT"].values, bounds = ([0,df_plot.loc[df_plot["Variant"]=="WT","IL"].min()/10], [df_plot.loc[df_plot["Variant"]=="WT","pSTAT"].max(), df_plot.loc[df_plot["Variant"]=="WT","IL"].max()*10]))
    mut_WT = fit[0]
    df_EC50.loc[len(df_EC50)] = [plot_num,df_plot["Cell_type"].values[0],"WT",df_plot["STAT_type"].values[0],np.log10(fit[1]),fit[0]/mut_WT*100]
    df.loc[df["Plot"]==plot_num,"pSTAT"] = df.loc[df["Plot"]==plot_num,"pSTAT"]/mut_WT*100
    for mutant in mutant_list:
        fit, cov = curve_fit(hill_func, df_plot.loc[df_plot["Variant"]==mutant,"IL"].values, df_plot.loc[df_plot["Variant"]==mutant,"pSTAT"].values, bounds = ([0,df_plot.loc[df_plot["Variant"]==mutant,"IL"].min()/10], [df_plot.loc[df_plot["Variant"]==mutant,"pSTAT"].max(), df_plot.loc[df_plot["Variant"]==mutant,"IL"].max()*10]))
        df_EC50.loc[len(df_EC50)] = [plot_num,df_plot["Cell_type"].values[0],mutant,df_plot["STAT_type"].values[0],np.log10(fit[1]),fit[0]/mut_WT*100]
df.to_csv("data/signaling/IL10M_STAT_data_Gorby.tsv.gz", sep="\t", index=False, compression="gzip") # After normalizing valeus to the amplitude we save the signaling dataset
df_EC50.to_csv("data/signaling/IL10M_EC50_AMP_Gorby.tsv.gz", sep="\t", index=False, compression="gzip")

## Expression data

#### Add YT cells

In [41]:
# Main dataset
df = pd.read_csv('../Data_analysis/data/whole_dataset.csv')

In [42]:
# Data for DERL2 cells
df2 = pd.read_csv('data/expression/raw/GSM2837061_DERL2.txt.gz', sep='\t', compression='gzip')
df2["RPK"] = df2["counts"].astype(int)/(abs(df2["end_position"].astype(int)-df2["start_position"].astype(int)+1)/1e3)
df2["Log2 TPM"] = np.log2(df2["RPK"]/(df2["RPK"].sum()/1e6)+1)
df2["external_gene_name"] = [name.replace("/","").replace(".","").replace(" ","").replace("-","").upper() for name in df2["external_gene_name"]]

# Get convertion to uniprot
df_uniprot = pd.read_csv('data/expression/raw/uniprotkb_AND_model_organism_9606_2025_02_18.tsv.gz', sep='\t', compression='gzip')
df_uniprot.index = [name.replace("/","").replace(".","").replace(" ","").replace("-","").upper() for name in df_uniprot["Gene Names (primary)"].astype("str")]
df_uniprot = df_uniprot[~df_uniprot.index.duplicated(keep='first')]
UP_dict = df_uniprot['Entry Name'].to_dict()

# Get uniprot entries into the RNAseq dataset and merge datasets of GEO and Uniprot
df2 = df2.loc[df2["external_gene_name"].isin(UP_dict.keys())]
df2["Gene"] = [UP_dict[name] for name in df2["external_gene_name"]]
df3 = df.loc[df["Cell"]=="ACH-001057"] # Get data in CCLE

df_DERL2 = pd.merge(df3, df2, on="Gene", how="inner")[["Gene","Log2 TPM_x","Log2 TPM_y"]]
df_DERL2.columns = ["Gene","Log2 TPM CCLE","Log2 TPM GEO"]
(df_DERL2["Log2 TPM CCLE"]-df_DERL2["Log2 TPM GEO"]).mean()

1.1995646787187606

In [43]:
# Data for KHYG1 cells
df2 = pd.read_csv('data/expression/raw/GSM2837065_KHYG1.txt.gz', sep='\t', compression='gzip')
df2["RPK"] = df2["counts"].astype(int)/(abs(df2["end_position"].astype(int)-df2["start_position"].astype(int)+1)/1e3)
df2["Log2 TPM"] = np.log2(df2["RPK"]/(df2["RPK"].sum()/1e6)+1)
df2["external_gene_name"] = [name.replace("/","").replace(".","").replace(" ","").replace("-","").upper() for name in df2["external_gene_name"]]

# Get uniprot entries into the RNAseq dataset and merge datasets of GEO and Uniprot
df2 = df2.loc[df2["external_gene_name"].isin(UP_dict.keys())]
df2["Gene"] = [UP_dict[name] for name in df2["external_gene_name"]]
df3 = df.loc[df["Cell"]=="ACH-001100"] # Get data in CCLE

df_KHYG1 = pd.merge(df3, df2, on="Gene", how="inner")[["Gene","Log2 TPM_x","Log2 TPM_y"]]
df_KHYG1.columns = ["Gene","Log2 TPM CCLE","Log2 TPM GEO"]
(df_KHYG1["Log2 TPM CCLE"]-df_KHYG1["Log2 TPM GEO"]).mean()

0.7729782647080803

In [44]:
# We get the bias between CCLE data and GEO data in the cells we have both data
bias = np.mean(np.array(df_KHYG1["Log2 TPM CCLE"].to_list()+df_DERL2["Log2 TPM CCLE"].to_list())-np.array(df_KHYG1["Log2 TPM GEO"].to_list()+df_DERL2["Log2 TPM GEO"].to_list()))

In [45]:
# Data for df2YT cells
df2 = pd.read_csv('data/expression/raw/GSM2837070_YT.txt.gz', sep='\t', compression='gzip')
df2["RPK"] = df2["counts"].astype(int)/(abs(df2["end_position"].astype(int)-df2["start_position"].astype(int)+1)/1e3)
df2["Log2 TPM"] = np.log2(df2["RPK"]/(df2["RPK"].sum()/1e6)+1)+bias
df2["external_gene_name"] = [name.replace("/","").replace(".","").replace(" ","").replace("-","").upper() for name in df2["external_gene_name"]]

# Get uniprot entries into the RNAseq dataset and merge datasets of GEO and Uniprot
df2 = df2.loc[df2["external_gene_name"].isin(UP_dict.keys())]
df2["Gene"] = [UP_dict[name] for name in df2["external_gene_name"]]

In [46]:
df_YT = df2[["Gene","Log2 TPM"]]
df_YT["Dataset type"] = "Cancer"
df_YT["RNA dataset num."] = 4
df_YT["Prot. count dataset num."] = np.nan
df_YT["Cell"] = "ACH-002317"
df_YT["Log10 Prot. count"] = np.nan
df_YT = df_YT[['Dataset type', 'RNA dataset num.', 'Prot. count dataset num.', 'Cell', 'Gene', 'Log2 TPM', 'Log10 Prot. count']]
df = pd.concat([df, df_YT], ignore_index=True)

#### Add MC/9 cells

In [47]:
df_MC9 = pd.read_csv('data/expression/raw/GSE98510_MC9.txt.gz', sep='\t', compression='gzip')
df_MC9["Gene"] = [name.replace("/","").replace(".","").replace(" ","").replace("-","").upper() for name in df_MC9["Unnamed: 0"]]
df_length = pd.read_csv('data/expression/raw/ncbi_dataset.tsv', sep='\t')
df_length["Gene"] = [name.replace("/","").replace(".","").replace(" ","").replace("-","").upper() for name in df_length["Symbol"]]
df_length = df_length[["Gene","Annotation Genomic Range Start","Annotation Genomic Range Stop","SwissProt Accessions"]]
df_length = df_length.drop_duplicates(subset=["Gene"], keep="first")
# Merge data anc calculate TPM
df_MC9 = pd.merge(df_MC9, df_length, on="Gene", how="inner").dropna()
df_MC9["RPK_1"] = df_MC9["C_1"].astype(int)/(abs(df_MC9["Annotation Genomic Range Stop"].astype(int)-df_MC9["Annotation Genomic Range Start"].astype(int)+1)/1e3)
df_MC9["C_1"] = np.log2(df_MC9["RPK_1"]/(df_MC9["RPK_1"].sum()/1e6)+1)
df_MC9["RPK_2"] = df_MC9["C_2"].astype(int)/(abs(df_MC9["Annotation Genomic Range Stop"].astype(int)-df_MC9["Annotation Genomic Range Start"].astype(int)+1)/1e3)
df_MC9["C_2"] = np.log2(df_MC9["RPK_2"]/(df_MC9["RPK_2"].sum()/1e6)+1)
df_MC9["RPK_3"] = df_MC9["C_3"].astype(int)/(abs(df_MC9["Annotation Genomic Range Stop"].astype(int)-df_MC9["Annotation Genomic Range Start"].astype(int)+1)/1e3)
df_MC9["C_3"] = np.log2(df_MC9["RPK_3"]/(df_MC9["RPK_3"].sum()/1e6)+1)
df_MC9["Log2 TPM"] = df_MC9[["C_1","C_2","C_3"]].mean(axis=1)
df_uniprot.index = df_uniprot["Entry"]
UP_dict2 = df_uniprot['Entry Name'].to_dict()
df_MC9 = df_MC9.loc[df_MC9["SwissProt Accessions"].isin(UP_dict2.keys())]
df_MC9["Gene"] = [UP_dict2[name] for name in df_MC9["SwissProt Accessions"]]
df_MC9 = df_MC9[["Gene","Log2 TPM"]]
df_MC9["Dataset type"] = "Mouse"
df_MC9["RNA dataset num."] = 5
df_MC9["Prot. count dataset num."] = np.nan
df_MC9["Cell"] = "MC/9"
df_MC9["Log10 Prot. count"] = np.nan
df_MC9 = df_MC9[['Dataset type', 'RNA dataset num.', 'Prot. count dataset num.', 'Cell', 'Gene', 'Log2 TPM', 'Log10 Prot. count']]
df = pd.concat([df, df_MC9], ignore_index=True)

#### Add BLaER1 cells

In [48]:
df_B = pd.read_csv('data/expression/raw/GSE182753_BLaER1.tsv.gz', sep='\t', compression='gzip')
df_E = pd.read_csv('data/expression/raw/Ensembl_human_genes.csv')
df_E.index = df_E["gene_id"]
E_dict = df_E['gene_name'].to_dict()
df_B["Gene"] = df_B.index
df_B = df_B.loc[df_B["Gene"].isin(E_dict.keys())]
df_B["Gene"] = [E_dict[gene].replace("/","").replace(".","").replace(" ","").replace("-","").upper() for gene in df_B["Gene"]]
df_B = df_B.loc[df_B["Gene"].isin(UP_dict.keys())]
df_B["Gene"] = [UP_dict[gene] for gene in df_B["Gene"]]
df_B["Log2 TPM"] = np.log2(df_B[["MG1","MG5"]].mean(axis=1)+1)
df_B["Dataset type"] = "Cancer"
df_B["RNA dataset num."] = 6
df_B["Prot. count dataset num."] = np.nan
df_B["Cell"] = "BLaER1"
df_B["Log10 Prot. count"] = np.nan
df_B = df_B[['Dataset type', 'RNA dataset num.', 'Prot. count dataset num.', 'Cell', 'Gene', 'Log2 TPM', 'Log10 Prot. count']]
df = pd.concat([df, df_B], ignore_index=True)

#### Add Haftl cells

In [49]:
df_H = pd.read_csv('data/expression/raw/GSE221737_Haftl.txt.gz', sep='\t', compression='gzip')
df_H["Gene"] = [gene.replace("/","").replace(".","").replace(" ","").replace("-","").upper() for gene in df_H["id"]]
df_H = df_H.loc[df_H["Gene"].isin(UP_dict.keys())]
df_H["Gene"] = [UP_dict[gene] for gene in df_H["Gene"]]
df_H["Log2 TPM"] = np.log2(df_H[["S1","S2","S9"]]).mean(axis=1) # Don't add +1 as expression of IL10RA and Il10RB is low -> leads to overestimation!
df_H["Dataset type"] = "Cancer"
df_H["RNA dataset num."] = 7
df_H["Prot. count dataset num."] = np.nan
df_H["Cell"] = "Haftl"
df_H["Log10 Prot. count"] = np.nan
df_H = df_H[['Dataset type', 'RNA dataset num.', 'Prot. count dataset num.', 'Cell', 'Gene', 'Log2 TPM', 'Log10 Prot. count']]
df = pd.concat([df, df_H], ignore_index=True)

In [ ]:
df.to_csv("data/expression/whole_dataset.tsv.gz", sep="\t", index=False, compression="gzip")

### Compare results between train and test

In [179]:
# Get training data, compute response and get MAE to compare with test data
df = pd.read_csv("data/expression/whole_dataset_IL10.tsv.gz", sep="\t", compression="gzip")
df_train = df.loc[df["Prot. count dataset num."]!=0]
df_train["Log10 Prot. count (estim.)"] = [df_lin.loc[df_lin["Gene"]==df_train.loc[index,"Gene"],"Intercept"].values[0]+df_train.loc[index,"Log2 TPM"]*df_lin.loc[df_lin["Gene"]==df_train.loc[index,"Gene"],"Slope"].values[0] for index in df_train.index]
df_train["MAE"] = abs(df_train["Log10 Prot. count"]-df_train["Log10 Prot. count (estim.)"])

In [207]:
f_statistic, p_value = stats.ttest_ind(df_test["MAE"].dropna().values, df_train["MAE"].dropna().values, equal_var=False, alternative="greater")
print("Train vs Test linear models p value: "+str(p_value))

Train vs Test linear models p value: 4.817688985315693e-13


### Immune dictionary data

In [32]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from scipy.optimize import curve_fit
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)
from sklearn.linear_model import TheilSenRegressor
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

In [40]:
df = pd.read_csv('data/expression/immune_dict_dataset_IL10.tsv.gz', sep='\t', compression='gzip')
df_S1 = df.loc[df["Gene"]=="STAT3_HUMAN"]
df_S1["Gene"] ="STAT1_HUMAN"
df = pd.concat([df, df_S1], ignore_index=True)
df_lin = pd.read_csv('data/lin_models/linear_models_IL10.csv')

In [42]:
for gene in df_lin["Gene"]:
    df.loc[(df["Gene"]==gene)&(df["Prot. count dataset num."].isna()),"Log10 Prot. count"] = df_lin.loc[df_lin["Gene"]==gene,"Intercept"].values[0]+df.loc[(df["Gene"]==gene)&(df["Prot. count dataset num."].isna()),"Log2 TPM"]*df_lin.loc[df_lin["Gene"]==gene,"Slope"].values[0]
    df.loc[(df["Gene"]==gene)&(df["Prot. count dataset num."].isna()),"Prot. count dataset num."] = 0

In [43]:
df.to_csv('data/expression/immune_dict_dataset_IL10.tsv.gz', sep="\t", index=False, compression="gzip") 